# Regime-Aware Energy/Commodity Quickstart

**Use case:** energy/commodity price forecasting, where a single averaged error metric hides that the backtest period covered several different market regimes.

Synthetic-only: `forecastlens.synthetic` generates regime-switching price series with known ground-truth regime changes, so `regime/` and `decision/` can be validated against a controlled series instead of a real one whose true regime changes nobody can label with certainty.

This notebook: builds a simple rolling-normal probabilistic forecast -> compares `VolatilityRegimeDetector` and `CUSUMDetector` via `RegimeAwareEvaluator` -> ends with `ProcurementTimingModel`'s economic-value takeaway.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

from forecastlens.core import ForecastResult
from forecastlens.decision import ProcurementTimingModel, ThresholdTimingRule
from forecastlens.evaluators import RegimeAwareEvaluator
from forecastlens.regime import CUSUMDetector, VolatilityRegimeDetector
from forecastlens.synthetic import load_preset

## 1. A three-regime Markov-switching price series

`regime_switch_markov` cycles between low/mid/high level-and-volatility regimes with a known transition matrix -- `series.changepoints` is the exact ground truth.

In [2]:
series = load_preset('regime_switch_markov')
values, timestamps = series.values, series.timestamps
print(f'{len(values)} periods, {len(series.changepoints)} true regime changes')

600 periods, 44 true regime changes


## 2. A causal rolling-normal probabilistic forecast

No trained model -- a transparent baseline: at each period, take the rolling mean/std of the *preceding* `window` periods (shifted by one, so it never sees the value it's forecasting) and treat the forecast as Gaussian around that.

In [3]:
window = 20
shifted = pd.Series(values).shift(1)
roll_mean = shifted.rolling(window, min_periods=window).mean()
roll_std = shifted.rolling(window, min_periods=window).std()

valid = (roll_mean.notna() & roll_std.notna() & (roll_std > 0)).to_numpy()
idx = np.flatnonzero(valid)
mu, sigma = roll_mean.to_numpy()[idx], roll_std.to_numpy()[idx]

levels = [0.1, 0.5, 0.9]
quantiles = {level: stats.norm.ppf(level, loc=mu, scale=sigma) for level in levels}
forecast = ForecastResult(timestamps=timestamps[idx], freq='D', quantiles=quantiles)
y_true = values[idx]
print(f'{len(idx)} periods with a valid forecast (first {window} are warmup)')

580 periods with a valid forecast (first 20 are warmup)


## 3. Regime-conditional CRPS: two swappable detectors

`RegimeAwareEvaluator` takes any `RegimeDetector` -- swapping detectors changes *which* structure the regime breakdown surfaces, which is itself worth seeing.

In [4]:
detectors = {
    'VolatilityRegimeDetector': VolatilityRegimeDetector(
        short_window=10, long_window=60, threshold_multiplier=1.3
    ),
    'CUSUMDetector': CUSUMDetector(warmup=60, threshold_std=8.0, drift_std=1.0),
}

for name, detector in detectors.items():
    report = RegimeAwareEvaluator(detector).evaluate(forecast, y_true)
    crps_by_regime = [r.crps for r in report.per_regime]
    print(f'--- {name} ---')
    print(f'  detected changepoints: {len(report.changepoints)}')
    print(f'  overall CRPS: {report.overall_crps:.3f}')
    print(f'  per-regime CRPS range: {min(crps_by_regime):.3f} - {max(crps_by_regime):.3f}')

--- VolatilityRegimeDetector ---
  detected changepoints: 18
  overall CRPS: 3.586
  per-regime CRPS range: 3.434 - 5.009
--- CUSUMDetector ---
  detected changepoints: 18
  overall CRPS: 3.586
  per-regime CRPS range: 2.246 - 15.906


Both detectors find a similar *number* of changepoints here, but the per-regime CRPS spread they surface differs a lot -- CUSUM's level-based changepoints isolate some genuinely harder-to-forecast segments (CRPS up to ~16) that the volatility detector's segmentation averages back down. **The regime-aware breakdown is only as informative as the regime definition feeding it** -- this is a diagnostic tool, not an oracle.

## 4. From diagnostics to a decision: is a forecast worth waiting on?

`ProcurementTimingModel` turns the same kind of forecast into a euro-per-unit decision, using `supply_shock_spike` -- a sudden price spike that mean-reverts.

In [5]:
shock_series = load_preset('supply_shock_spike')
start, horizon = 30, 30  # the procurement window opens right at the spike's peak
baseline_forecast = shock_series.values[:20].mean()  # expect reversion to the pre-shock level
shock_forecast = ForecastResult(
    timestamps=shock_series.timestamps[start : start + horizon],
    freq='D',
    point=np.full(horizon, baseline_forecast),
)
realized_prices = shock_series.values[start : start + horizon]

model = ProcurementTimingModel(
    decision_rule=ThresholdTimingRule(wait_if_forecast_below_current_by=0.03),
    deadline_horizon=horizon,
    units=1000,
)
report = model.evaluate(shock_forecast, realized_prices)
print(f'Bought on day {report.buy_day} (forced by deadline: {report.forced_by_deadline})')
print(f'Savings per unit: {report.savings_per_unit:.2f}')
print(f'Total savings:    {report.total_savings:.2f}')

Bought on day 11 (forced by deadline: False)
Savings per unit: 21.94
Total savings:    21940.29


Waiting for the price to revert toward the forecast baseline instead of buying at the spike -- expressed directly in currency, which is the whole point of the economic-value layer: no CRPS/WQL vocabulary needed to explain the result to a procurement stakeholder.